# PyMessage Walkthrough

A hands-on tour of every function in the package, running entirely against the **built-in example backup** — no real iPhone needed.

Each section is independent. Run them in order for the smoothest experience, or jump straight to whatever interests you.

In [ ]:
from pymessage import (
    EXAMPLE_BACKUP,
    Backup,
    find_backups,
    get_activity_summary,
    get_attachments,
    get_contact_summary,
    get_messages,
)

---
## 1. Discovering Backups — `find_backups()`

`find_backups()` scans two places on your Mac:
- `~/Library/Application Support/MobileSync/Backup/` — iPhone backups
- `~/Library/Messages/chat.db` — macOS Messages app

It returns a plain `list[Backup]` sorted by most recent first.

In [ ]:
backups = find_backups()

if backups:
    for b in backups:
        print(b)
else:
    print("No backups found on this machine.")

The `Backup` dataclass wraps everything needed to open a database:

| Field | Description |
|-------|-------------|
| `type` | `"iphone"` or `"macos"` |
| `path` | Backup directory (iphone) or path to `chat.db` (macos) |
| `device_name` | Human-readable label |
| `last_backup` | Datetime of most recent backup |
| `ios_version` | e.g. `"17.2"` |
| `phone_number` | Device phone number |

`EXAMPLE_BACKUP` is a fake iPhone backup built into the package for testing and demos:

In [ ]:
print(EXAMPLE_BACKUP)
print()
print(f"type         = {EXAMPLE_BACKUP.type!r}")
print(f"device_name  = {EXAMPLE_BACKUP.device_name!r}")
print(f"ios_version  = {EXAMPLE_BACKUP.ios_version!r}")
print(f"phone_number = {EXAMPLE_BACKUP.phone_number!r}")
print(f"last_backup  = {EXAMPLE_BACKUP.last_backup}")
print(f"path         = {EXAMPLE_BACKUP.path}")

---
## 2. Getting Messages — `get_messages()`

`get_messages(backup)` queries the iMessage database and returns a DataFrame where every row is one message (or tapback reaction).

In [ ]:
df = get_messages(EXAMPLE_BACKUP)

df.head()

### `contact_name` — display names from the handle table

- Sent messages (`is_from_me=True`) → `"Me"`
- Received messages → resolved from `handle.uncanonicalized_id` (the display name iMessage stores)

In [ ]:
sent     = df[df["is_from_me"]]
received = df[~df["is_from_me"]]

print(f"Sent messages:     {len(sent):>3}  (contact_name = 'Me')")
print(f"Received messages: {len(received):>3}  (contact_name = sender's display name)")
print()
print("All contacts:", sorted(df["contact_name"].unique().tolist()))

### Tapback reactions

Reactions are stored as separate messages in iMessage. `reaction_type` and `reaction_action` are parsed from `associated_message_type` (codes 2000–2007 = add, 3000–3007 = remove).

In [ ]:
reactions = df[df["reaction_type"].notna()][
    ["contact_name", "reaction_type", "reaction_action", "message_text"]
]
reactions

---
## 3. Filtering Messages

All filters are optional and composable.

### Filter by phone number

In [ ]:
caleb_df = get_messages(EXAMPLE_BACKUP, phone_numbers="+18015550002")

print(f"Messages with Caleb: {len(caleb_df)}")
caleb_df[["timestamp", "contact_name", "message_text"]].head(6)

### Filter by multiple contacts

In [ ]:
multi_df = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers=["+18015550002", "+18015550003"],  # Caleb + Mitch
)
print(f"Messages with Caleb or Mitch: {len(multi_df)}")
multi_df["contact_name"].value_counts()

### Filter by date range

In [ ]:
# Build a range from the second half of the data
all_ts   = df["timestamp"]
midpoint = all_ts.min() + (all_ts.max() - all_ts.min()) / 2
start    = midpoint.strftime("%Y-%m-%d")
end      = all_ts.max().strftime("%Y-%m-%d")

recent_df = get_messages(EXAMPLE_BACKUP, date_range=(start, end))

print(f"Full data:  {all_ts.min().date()} → {all_ts.max().date()}  ({len(df)} messages)")
print(f"Filtered:   {start} → {end}  ({len(recent_df)} messages)")
recent_df[["timestamp", "contact_name", "message_text"]].head(5)

### Combine filters — phone number + date range

In [ ]:
filtered = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers="+18015550002",
    date_range=(start, end),
)
print(f"Caleb messages in date range: {len(filtered)}")
filtered[["timestamp", "contact_name", "message_text"]]

---
## 4. Getting Attachments — `get_attachments()`

Returns metadata for every file shared in messages. For iPhone backups, `file_path` resolves the SHA-1 hashed path inside the backup directory.

In [ ]:
attachments = get_attachments(EXAMPLE_BACKUP)

print(f"Attachments: {len(attachments)}")
print(f"Columns: {list(attachments.columns)}")
attachments

In [ ]:
# Filter attachments to a specific contact
att_filtered = get_attachments(EXAMPLE_BACKUP, phone_numbers="+18015550002")
print(f"Attachments from Caleb: {len(att_filtered)}")

---
## 5. Activity Summary — `get_activity_summary()`

Computes overall statistics across all messages. Returns `(summary_df, top_contacts_df)`.

Optional: `start`, `end`, `last_n_days`, `reference_date`, `top_n`

In [ ]:
summary, top_contacts = get_activity_summary(df)

s = summary.iloc[0]
s

In [ ]:
print("=== Top Contacts ===")
top_contacts

### Filter to last N days

In [ ]:
import pandas as pd

# Use the data's own max timestamp as the reference so the example always works
ref = df["timestamp"].max()

summary_30, top_30 = get_activity_summary(df, last_n_days=30, reference_date=ref)
print(f"Messages in last 30 days: {summary_30.iloc[0]['total_messages']}")
top_30

---
## 6. Per-Contact Stats — `get_contact_summary()`

Returns a single-row DataFrame with detailed stats for one conversation. Accepts the phone number in any format.

In [ ]:
cs = get_contact_summary(df, "+18015550002").iloc[0]  # Caleb
cs

In [ ]:
# Mitch 
ms = get_contact_summary(df, "+18015550003").iloc[0]
ms

---
## Using Your Own Data

Everything above works identically against a real iPhone backup or macOS Messages database. Just swap `EXAMPLE_BACKUP` for your own:

In [ ]:
# Uncomment and run to use your real data:

from pymessage import find_backups, get_messages
backups = find_backups()
print(backups)          # pick the one you want

df = get_messages(backups[0])
df.head()

---
## Quick Reference

| Function | Returns | Key optional params |
|----------|---------|---------------------|
| `find_backups()` | `list[Backup]` | — |
| `get_messages(backup)` | `DataFrame` | `phone_numbers`, `date_range`, `output_csv` |
| `get_attachments(backup)` | `DataFrame` | `phone_numbers` |
| `get_activity_summary(df)` | `(summary_df, top_df)` | `start`, `end`, `last_n_days`, `top_n` |
| `get_contact_summary(df, contact)` | `DataFrame` | `start`, `end`, `last_n_days` |

In [ ]:
get_messages('/Users/tuckertrost/Library/Messages/chat.db')